# 03. 가상 당도 생성과 결합 모델 실험

가상 당도는 실측이 아닙니다. RGB 대리값 생성 → A/B 실험 → 가상 상품등급 학습·CV → 재사용 Test 진단 순서입니다. 진단 결과를 독립 최종 성능으로 해석하지 않습니다.

실행 결과는 저장하지 않았습니다. 아래 실행 스위치는 기본 `False`이며, 전체 실행만으로 학습·파일 생성·서버 검사가 시작되지 않습니다.

In [ ]:
from pathlib import Path
import os
import sys
import json

# 저장소 루트 또는 notebooks 폴더에서 실행할 수 있습니다.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/data').is_dir() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('CQC 저장소 안에서 노트북을 열어 주세요.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('작업 폴더:', ROOT)


## 가상 당도 생성

보정에는 개발 사과만 사용합니다. 0.34는 생성 설정이지 실측 당도 예측 정확도가 아닙니다.

In [ ]:
from src.data.multiview import load_groups
from src.data.virtual_brix import extract_group_features, build_virtual_brix_rows, write_virtual_brix

RUN_GENERATE_BRIX = False
if RUN_GENERATE_BRIX:
    destination = Path('outputs/notebook-virtual-brix.csv')
    if destination.exists():
        raise FileExistsError(destination)
    groups = load_groups(Path('data/processed/manifest.csv'), Path('configs/splits/seed-42.csv'))
    features = extract_group_features(groups, Path('data/raw'), views=12)
    development = {group.group_no for group in groups if group.split != 'test'}
    rows = build_virtual_brix_rows(features, target_correlation=0.34, fit_group_numbers=development)
    write_virtual_brix(destination, rows)
    print('생성 사과 수:', len(rows), '/ 실측: False')


## 당도 결합 A/B 실험

기존 완료 실험은 건너뛰는 공통 실행기를 사용합니다. 생성 CSV를 바꾸려면 경로도 함께 변경하세요.

In [ ]:
from src.training import brix_experiments, brix_report

RUN_BRIX_AB = False
RUN_BRIX_REPORT = False
if RUN_BRIX_AB:
    brix_experiments.main(['--virtual-brix', 'data/processed/virtual-brix.csv', '--device', 'cuda', '--execute'])
if RUN_BRIX_REPORT:
    brix_report.main([])


## 가상 상품등급 학습

이미지 외관등급이 아닌 고정 정책의 합성 상품등급을 학습합니다.

In [ ]:
"""외관 등급과 가상 당도로 만든 시연용 상품등급을 학습한다."""
from collections import Counter
from dataclasses import replace
from pathlib import Path
import hashlib

import torch
from torch.utils.data import DataLoader

from src.data.multiview import load_groups, MultiViewDataset
from src.data.torch_dataset import TorchMultiViewDataset
from src.data.virtual_brix import load_virtual_brix
from src.inference.commercial_policy import assess_commercial_grade, POLICY_VERSION
from src.training.models import build_model
from src.training.engine import run_epoch, save_checkpoint
from src.training.train import set_reproducibility, write_json, validation_score


In [ ]:
def train_commercial(*, output_dir=Path('outputs/training-commercial-v3'), epochs=25, validation_scheme='source', cv_fold=0, execute=False):
    """시험 데이터를 제외하고 고정 정책의 가상 상품등급 모델을 학습한다."""
    if epochs < 1 or output_dir.exists():
        raise ValueError('epochs must be positive and output directory must not exist')
    set_reproducibility(42)
    manifest = Path('data/processed/manifest.csv')
    splits = Path('configs/splits/seed-42.csv')
    brix_path = Path('data/processed/virtual-brix.csv')
    brix = load_virtual_brix(brix_path)
    original = [g for g in load_groups(manifest, splits) if g.split != 'test']
    groups, labels = [], []
    for group in original:
        value = brix[group.group_no][0]
        result = assess_commercial_grade(group.quality_grade, value)
        grade = result['commercial_grade']
        if grade is None:
            raise ValueError(f'No commercial target: {group.group_no}')
        groups.append(replace(group, quality_grade=grade))
        labels.append(dict(group_no=group.group_no, original_split=group.original_split,
                           appearance_grade=group.quality_grade, virtual_brix=value,
                           commercial_grade=grade, score=result['score']))
    if validation_scheme == 'source':
        train = [g for g in groups if g.original_split == 'train']
        validation = [g for g in groups if g.original_split == 'validation']
    else:
        train = [g for g in groups if g.cv_fold != cv_fold]
        validation = [g for g in groups if g.cv_fold == cv_fold]
    counts = {name: dict(Counter(g.quality_grade for g in data))
              for name, data in [('train', train), ('validation', validation)]}
    if any(set(count) != {'L', 'M', 'S'} for count in counts.values()):
        raise ValueError(f'Each development subset needs all grades: {counts}')
    config = dict(model_version='commercial-v3-candidate', model_kind='separate_brix',
                  target_semantics='simulated_commercial_grade', policy_version=POLICY_VERSION,
                  brix_is_measured=False, views=12, image_size=224, seed=42,
                  epochs=epochs, batch_size=2, workers=0, dropout=0.4,
                  learning_rate=0.0003, weight_decay=0.0005, quality_loss='focal',
                  focal_gamma=2.0, train_groups=len(train), validation_groups=len(validation),
                  validation_scheme=validation_scheme, cv_fold=cv_fold if validation_scheme == 'cv' else None,
                  view_sampling='fixed', test_used=False,
                  class_counts=counts, virtual_brix=str(brix_path),
                  source_hashes={str(p): hashlib.sha256(p.read_bytes()).hexdigest()
                                 for p in (manifest, splits, brix_path)},
                  limitation='Agreement with synthetic policy targets, not measured commercial quality')
    print(config, flush=True)
    if not execute:
        return
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA required for this remote training run')
    output_dir.mkdir(parents=True, exist_ok=False)
    write_json(output_dir / 'config.json', config)
    write_json(output_dir / 'development-labels.json', labels)
    device = torch.device('cuda')
    sources = [MultiViewDataset(data, Path('data/raw'), 12) for data in (train, validation)]
    loaders = [DataLoader(TorchMultiViewDataset(source, training=i == 0,
                        image_size=224, virtual_brix=brix), batch_size=2,
                        shuffle=i == 0, num_workers=0,
                        generator=torch.Generator().manual_seed(42))
               for i, source in enumerate(sources)]
    model = build_model('separate_brix', pretrained=True, dropout=0.4).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003, weight_decay=0.0005)
    history, best = [], -1.0
    try:
        for epoch in range(1, epochs + 1):
            print(f'epoch_start={epoch}/{epochs}', flush=True)
            train_metrics = run_epoch(model, loaders[0], device, optimizer=optimizer,
                                      quality_loss_kind='focal', focal_gamma=2.0)
            metrics = run_epoch(model, loaders[1], device, quality_loss_kind='focal', focal_gamma=2.0)
            score = validation_score(metrics)
            history.append(dict(epoch=epoch, train=train_metrics, validation=metrics, validation_score=score))
            write_json(output_dir / 'history.json', history)
            for name in (['last.pt', 'best.pt'] if score > best else ['last.pt']):
                save_checkpoint(output_dir / name, model=model, optimizer=optimizer,
                                epoch=epoch, config=config, metrics=metrics)
            best = max(best, score)
            print(f'epoch={epoch}/{epochs} commercial_macro_f1={metrics["quality"]["macro_f1"]:.6f}', flush=True)
    finally:
        for source in sources:
            source.close()
    write_json(output_dir / 'summary.json', dict(epochs_completed=len(history),
               best_validation_score=best, test_used=False, target_semantics=config['target_semantics']))


In [ ]:
RUN_TRAIN_COMMERCIAL = False
if RUN_TRAIN_COMMERCIAL:
    train_commercial(output_dir=Path("outputs/notebook-commercial-v3"), execute=True)


## 사과 단위 5-fold CV

위 학습 함수를 fold별로 호출합니다. 사전 선택한 17 epoch를 그대로 비교합니다.

In [ ]:
"""고정 상품등급 정책으로 사과 단위 5개 교차검증 학습을 순서대로 실행한다."""
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path


In [ ]:
def run_commercial_cv(*, root=Path('outputs/training-commercial-v3-cv'), epochs=25):
    """사과 단위 5개 교차검증을 실행하고 사전 지정한 17회차 결과를 비교한다."""
    if epochs < 17:
        raise ValueError('At least 17 epochs needed for the preselected source epoch comparison')
    if root.exists():
        raise ValueError(f'Refusing to reuse output root: {root}')
    root.mkdir(parents=True)
    for fold in range(5):
        destination = root / f'fold-{fold}'
        print(f'cv_run={fold + 1}/5', flush=True)
        train_commercial(validation_scheme='cv', cv_fold=fold,
                         epochs=epochs, output_dir=destination, execute=True)
    rows = []
    for fold in range(5):
        history = json.loads((root / f'fold-{fold}' / 'history.json').read_text(encoding='utf-8'))
        selected = next(item for item in history if item['epoch'] == 17)
        rows.append(dict(fold=fold, groups=selected['validation']['samples'],
                         quality_macro_f1=selected['validation']['quality']['macro_f1'],
                         cultivar_macro_f1=selected['validation']['cultivar']['macro_f1']))
    from statistics import mean
    report = dict(evaluation_role='development_group_cv', fixed_epoch=17,
                  epoch_source='source_validation_preselected_before_cv', test_used=False,
                  target_semantics='simulated_commercial_grade', folds=rows,
                  mean_quality_macro_f1=mean(row['quality_macro_f1'] for row in rows),
                  mean_cultivar_macro_f1=mean(row['cultivar_macro_f1'] for row in rows))
    (root / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(f'cv_complete quality_macro_f1={report["mean_quality_macro_f1"]:.6f}', flush=True)


In [ ]:
RUN_RUN_COMMERCIAL_CV = False
if RUN_RUN_COMMERCIAL_CV:
    run_commercial_cv(root=Path("outputs/notebook-commercial-v3-cv"))


## 기존 모델 + 정책 기준선

이미 사용한 Test의 진단이며 새 승인 근거가 아닙니다.

In [ ]:
"""이미 사용한 시험 사과에서 v2 예측·고정 정책과 합성 상품등급의 일치도를 비교한다."""
from __future__ import annotations

import json
from pathlib import Path

import torch

from src.data.virtual_brix import load_virtual_brix
from src.inference.commercial_policy import assess_commercial_grade, POLICY_VERSION
from src.training.engine import classification_metrics

CLASSES = ('L', 'M', 'S')


In [ ]:
def evaluate_policy_baseline(*, v2_predictions=Path('docs/wbs/results/candidate-v2-reused-test-regression.json'), virtual_brix=Path('data/processed/virtual-brix.csv'), output=Path('docs/wbs/results/commercial-v2-rule-reused-test.json')):
    """기존 외관 예측과 가상 당도 정책의 상품등급 일치도를 진단한다."""
    if output.exists():
        raise ValueError(f'Refusing to overwrite: {output}')
    source = json.loads(v2_predictions.read_text(encoding='utf-8'))
    brix = load_virtual_brix(virtual_brix)
    matrix = torch.zeros((3, 3), dtype=torch.long)
    seen = set()
    for row in source['test']['predictions']:
        key = row['group_no']
        if key in seen:
            raise ValueError(f'Duplicate apple in Test: {key}')
        seen.add(key)
        value = brix[key][0]
        actual = assess_commercial_grade(CLASSES[row['quality_target_index']], value)['commercial_grade']
        predicted = assess_commercial_grade(CLASSES[row['quality_prediction_index']], value)['commercial_grade']
        matrix[CLASSES.index(actual), CLASSES.index(predicted)] += 1
    if len(seen) != 27:
        raise ValueError(f'Expected 27 apples, found {len(seen)}')
    result = dict(evaluation_role='reused_test_diagnostic_simulated_commercial_target',
                  independent_final_approval=False, may_select_model_or_epoch=False,
                  method='v2_appearance_prediction_plus_exact_demo_commercial_rule',
                  policy_version=POLICY_VERSION, brix_is_measured=False,
                  samples=len(seen), commercial=classification_metrics(matrix))
    output.write_text(json.dumps(result, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    print(f'commercial_macro_f1={result["commercial"]["macro_f1"]:.6f}')


In [ ]:
RUN_EVALUATE_POLICY_BASELINE = False
if RUN_EVALUATE_POLICY_BASELINE:
    evaluate_policy_baseline(output=Path("outputs/notebook-policy-baseline.json"))


## 결합 모델 진단

사전에 선택한 source checkpoint만 사용하며 평가 파일을 덮어쓰지 않습니다.

In [ ]:
"""이미 사용한 시험 사과에서 v3 예측을 시연용 상품등급 정답과 비교한다."""
from __future__ import annotations

from collections import Counter
from dataclasses import replace
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from src.data.multiview import MultiViewDataset, load_groups
from src.data.torch_dataset import TorchMultiViewDataset
from src.data.virtual_brix import load_virtual_brix
from src.inference.commercial_policy import assess_commercial_grade, POLICY_VERSION
from src.training.engine import run_epoch
from src.training.evaluate import load_checkpoint
from src.training.models import build_model
from src.training.train import write_json


In [ ]:
def evaluate_commercial(*, checkpoint, output):
    """미리 선택한 상품등급 모델을 재사용 시험 사과로 진단한다."""
    if output.exists():
        raise ValueError(f'Refusing to overwrite: {output}')
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    checkpoint_path = checkpoint
    checkpoint = load_checkpoint(checkpoint_path, device)
    config = checkpoint['config']
    if config.get('target_semantics') != 'simulated_commercial_grade' or config.get('policy_version') != POLICY_VERSION:
        raise ValueError('Checkpoint is not the approved simulated-commercial target definition')
    if config.get('validation_scheme') != 'source':
        raise ValueError('Reused Test diagnostic must use the preselected source checkpoint')
    brix = load_virtual_brix(Path('data/processed/virtual-brix.csv'))
    original = [g for g in load_groups(Path('data/processed/manifest.csv'),
                                       Path('configs/splits/seed-42.csv')) if g.split == 'test']
    groups = []
    for group in original:
        synthetic = assess_commercial_grade(group.quality_grade, brix[group.group_no][0])['commercial_grade']
        if synthetic is None:
            raise ValueError(f'Missing simulated target: {group.group_no}')
        groups.append(replace(group, quality_grade=synthetic))
    source = MultiViewDataset(groups, Path('data/raw'), 12)
    model = build_model('separate_brix', pretrained=False, dropout=config['dropout']).to(device)
    model.load_state_dict(checkpoint['model_state'])
    try:
        loader = DataLoader(TorchMultiViewDataset(source, training=False, image_size=224,
                                                  virtual_brix=brix), batch_size=2, num_workers=0)
        metrics = run_epoch(model, loader, device, quality_loss_kind='focal',
                            focal_gamma=2.0, include_predictions=True)
    finally:
        source.close()
    write_json(output, dict(
        evaluation_role='reused_test_diagnostic_simulated_commercial_target',
        independent_final_approval=False, may_select_model_or_epoch=False,
        checkpoint=str(checkpoint_path), checkpoint_epoch=checkpoint['epoch'],
        target_semantics=config['target_semantics'], policy_version=POLICY_VERSION,
        brix_is_measured=False, test_used_for_training=False,
        target_counts=dict(Counter(g.quality_grade for g in groups)), test=metrics))
    print(f'test_groups={len(groups)} commercial_macro_f1={metrics["quality"]["macro_f1"]:.6f}', flush=True)


In [ ]:
RUN_EVALUATE_COMMERCIAL = False
if RUN_EVALUATE_COMMERCIAL:
    evaluate_commercial(checkpoint=Path("outputs/training-commercial-v3/best.pt"), output=Path("outputs/notebook-commercial-diagnostic.json"))
